In [5]:
# ====================================
# Part 3 - Real Estate Data Scraper
# ====================================


# ====================================
# Imports
# ====================================

from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import requests


# ====================================
# Configuration
# ====================================

BASE_URL = "https://www.ceresne.sk"

FLATS_URL = (
    f"{BASE_URL}/app/uploads/flats.json"
)

DATA_DIR = Path("data")

DATA_DIR.mkdir(
    exist_ok=True
)

OUTPUT_FILE = (
    DATA_DIR / "03_ceresne_listings.csv"
)

CHARTS_DIR = Path("charts")


# ====================================
# Load apartment data
# ====================================

response = requests.get(
    FLATS_URL,
    timeout=30
)


print(response.status_code)
print(response.headers.get("Content-Type"))

response.raise_for_status()

flats_data = response.json()

print(
    f"Buildings found: {len(flats_data)}"
)


# ====================================
# Extract listings
# ====================================

listings = []

for building_data in flats_data.values():

    records = building_data.get(
        "records",
        {}
    )

    for record in records.values():

        listing = {
            "listing_id": record.get("flat"),
            "building": record.get("building"),
            "floor": record.get("floot"),
            "rooms": record.get("rooms"),
            "area": record.get("area"),
            "price": record.get("price"),
            "status": record.get("status"),
        }

        listing_id = record.get("flat")
        status = record.get("status")

        listing["url"] = (
            f"{BASE_URL}/ponuka-bytov/byt/"
            f"{listing_id.lower()}/"
            if listing_id and status == "Voľný"
            else None
        )

        listings.append(listing)


df = pd.DataFrame(listings)

print(
    f"Scraped listings: {len(df)}"
)


# ====================================
# Clean data
# ====================================

# Convert area to numeric
df["area_m2"] = (
    df["area"]
    .str.replace(" m²", "", regex=False)
    .str.replace(",", ".", regex=False)
)

df["area_m2"] = pd.to_numeric(
    df["area_m2"],
    errors="coerce"
)


# Convert price to numeric
df["price_eur"] = (
    df["price"]
    .str.replace(" €", "", regex=False)
    .str.replace(" ", "", regex=False)
)

df["price_eur"] = pd.to_numeric(
    df["price_eur"],
    errors="coerce"
)


# Convert rooms and floor to numeric
df["rooms"] = pd.to_numeric(
    df["rooms"],
    errors="coerce"
)

df["floor"] = pd.to_numeric(
    df["floor"],
    errors="coerce"
)


# Calculate price per m²
df["price_per_m2"] = (
    df["price_eur"]
    / df["area_m2"]
)


# ====================================
# Remove duplicates
# ====================================

duplicates = (
    df["listing_id"]
    .duplicated()
    .sum()
)

print(
    f"Duplicate listing IDs: {duplicates}"
)

df = df.drop_duplicates(
    subset="listing_id"
)

df = df.reset_index(
    drop=True
)


# ====================================
# Validation
# ====================================

print("\nValidation summary")
print("------------------")

print(
    f"Total listings: {len(df)}"
)

print(
    f"Missing listing IDs: "
    f"{df['listing_id'].isna().sum()}"
)

print(
    f"Missing prices: "
    f"{df['price_eur'].isna().sum()}"
)

print(
    f"Missing areas: "
    f"{df['area_m2'].isna().sum()}"
)

print(
    f"Duplicate IDs: "
    f"{df['listing_id'].duplicated().sum()}"
)

print(
    f"Duplicate URLs: "
    f"{df['url'].duplicated().sum()}"
)


# ====================================
# Listings by status
# ====================================

print("\nListings by status")
print("------------------")

status_counts = (
    df["status"]
    .value_counts(
        dropna=False
    )
)

print(status_counts)


# ====================================
# Basic analysis
# ====================================

print("\nBasic statistics")
print("----------------")

print(
    f"Average price: "
    f"{df['price_eur'].mean():,.0f} €"
)

print(
    f"Median price: "
    f"{df['price_eur'].median():,.0f} €"
)

print(
    f"Average area: "
    f"{df['area_m2'].mean():.2f} m²"
)

print(
    f"Average price per m²: "
    f"{df['price_per_m2'].mean():,.0f} €/m²"
)


# ====================================
# Create charts directory
# ====================================

CHARTS_DIR.mkdir(
    exist_ok=True
)


# ====================================
# Chart 1 - Listings by status
# ====================================

plt.figure()

status_counts.plot(
    kind="bar"
)

plt.title(
    "Listings by Status"
)

plt.xlabel(
    "Status"
)

plt.ylabel(
    "Number of Listings"
)

plt.xticks(
    rotation=45,
    ha="right"
)

plt.tight_layout()

plt.savefig(
    CHARTS_DIR / "listings_by_status.png"
)

plt.close()


# ====================================
# Chart 2 - Price by number of rooms
# ====================================

room_prices = (
    df.dropna(
        subset=["rooms", "price_eur"]
    )
    .groupby("rooms")["price_eur"]
    .mean()
)

plt.figure()

room_prices.plot(
    kind="bar"
)

plt.title(
    "Average Price by Number of Rooms"
)

plt.xlabel(
    "Number of Rooms"
)

plt.ylabel(
    "Average Price (€)"
)

plt.xticks(
    rotation=0
)

plt.tight_layout()

plt.savefig(
    CHARTS_DIR / "price_by_rooms.png"
)

plt.close()


# ====================================
# Chart 3 - Price distribution
# ====================================

plt.figure()

df["price_eur"].dropna().plot(
    kind="hist",
    bins=20
)

plt.title(
    "Distribution of Apartment Prices"
)

plt.xlabel(
    "Price (€)"
)

plt.ylabel(
    "Number of Listings"
)

plt.tight_layout()

plt.savefig(
    CHARTS_DIR / "price_distribution.png"
)

plt.close()


# ====================================
# Save CSV
# ====================================

df.to_csv(
    OUTPUT_FILE,
    index=False,
    encoding="utf-8-sig"
)

print(
    f"\nCSV saved to: {OUTPUT_FILE}"
)


# ====================================
# Finished
# ====================================

print(
    "\nScraping completed successfully."
)

200
application/json
Buildings found: 17
Scraped listings: 1033
Duplicate listing IDs: 0

Validation summary
------------------
Total listings: 1033
Missing listing IDs: 0
Missing prices: 944
Missing areas: 0
Duplicate IDs: 0
Duplicate URLs: 943

Listings by status
------------------
status
Predaný            709
Nedostupný         216
Voľný               89
Rezervovaný         12
Predrezervovaný      7
Name: count, dtype: int64

Basic statistics
----------------
Average price: 311,485 €
Median price: 269,721 €
Average area: 71.39 m²
Average price per m²: 4,845 €/m²

CSV saved to: data\03_ceresne_listings.csv

Scraping completed successfully.
